# GQS of two electrons on L sites: interaction sweep

Thin experiment notebook: all functions live in `chain_gqs`. Change parameters here only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from chain_gqs import ChainModel, townsend2021, coulomb, product_state, gqs_history, projected_ensemble, check_ensemble
from chain_gqs.plotting import plot_gqs_history, plot_sector_weights

## Parameters

In [ ]:
L, N = 5, 2
subsystem = (0, 1)
occupied = (0, 2)
dt, nsteps = 0.02, 300
method = 'trotter'   # 'exact' removes Trotter error

# Model: Townsend et al. parameters, swept in lam_ee/t.
model = townsend2021(L=L, N=N, lam_ee=1.0, t=1.0)
sweep = dict(lam_ee=[0.5, 2.0, 10.0])

# Original notebook model instead:
# model = coulomb(L=L, N=N, J=1.0, V=1.0)
# sweep = dict(J=[0.5, 1.0, 1.5])

## Single snapshot

In [ ]:
print(model.label)
times, Q = gqs_history(model, product_state(L, occupied), nsteps, dt, subsystem, method)
Q[-1].sector_table()

In [ ]:
# Same numbers, two languages: 'pe' (projected ensemble) or 'cm' (condensed matter)
display(Q[-1].ensemble_table(style='pe'))
display(Q[-1].ensemble_table(style='cm'))

In [ ]:
print('Sector probabilities:', Q[-1].sector_weights)
from chain_gqs.dynamics import evolve
check_ensemble(Q[-1], evolve(model, product_state(L, occupied), nsteps, dt, method)[-1])
print('Consistency checks passed')

## Sweep over J/V

In [ ]:
(key, values), = sweep.items()
results = {}
for v in values:
    m = model.with_(**{key: v})
    results[v] = gqs_history(m, product_state(L, occupied), nsteps, dt, subsystem, method)
    fig, _ = plot_gqs_history(results[v][1], bloch_view='angles', title=m.label)
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(values), figsize=(15, 3), sharey=True)
for ax, v in zip(axes, values):
    plot_sector_weights(*results[v], ax=ax)
    ax.set_title(f'{key}={v}')
plt.show()

## Measurement-based reconstruction (needs Qiskit)

In [ ]:
from chain_gqs.quantum import build_circuit, gqs_measurement_circuits, sample_counts
from chain_gqs import reconstruct_from_counts, require_pure_reconstruction

sample_steps = np.arange(0, 201, 5)
Q_measured = []
for i, n in enumerate(sample_steps):
    circ = build_circuit(model, int(n), dt, occupied)
    counts = sample_counts(gqs_measurement_circuits(circ, subsystem), shots=10_000, seed=100 + i)
    Q_measured.append(require_pure_reconstruction(
        reconstruct_from_counts(counts, L, subsystem, N, assume_pure=True)))
fig, _ = plot_gqs_history(Q_measured, bloch_view='angles')
plt.show()